# runtime comparison

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Record Zoetrope GPU timing and compare the frozen solver times

In [ ]:
# Qian PFC | Zoetrope / Spatial TI three-section GPU training + runtime comparison
# Frozen original section IDs [1, 2, 7]:
#   GW15 UMB1117/F1b (410,492), GW20 FB080/F1 (665,204), GW22 FB123/F2 (439,986).
# EXACT same default Stage-1a architecture/optimization as the PFC nine-section fit:
#   1,500 epochs, 1,024 cells/section/batch, seed 20260622, global gauge,
#   shared-auto coordinate decoder, library-size offset.
# Times the native run_stage1a(...) call (CUDA synchronized), saves an immutable receipt,
# then reads the ORIGINAL three-section comparator receipts and draws an editable plot.
# IMPORTANT: GW20 is INCLUDED in this runtime-training cohort. Never use this
# checkpoint for the strict GW20-held-out expression forecasting benchmark.
# FULL training needs a Slurm GPU node with CUDA-enabled PyTorch. No retraining baselines.

from pathlib import Path
from dataclasses import asdict, replace, fields
import os, sys, json, hashlib, time, platform, subprocess, inspect, random, gc
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt

# --------------------------- User settings ---------------------------
SMOKE = False            # True = 3 epochs, <=3,000 cells/section; no publication timing.
PROJECT = Path(os.getenv('STP_PROJECT',
    os.environ["STP_PROJECT"]))
DATA_ROOT = Path(os.getenv('QIAN_PFC_DATA_ROOT',
    os.environ["QIAN_PFC_DATA_ROOT"]))
REPO = Path(os.getenv('FLOW_STP_REPO',
    os.environ["FLOW_STP_REPO"]))
PINNED_COMMIT = '76bb952a841c3671ec35161149e780167fd9b7b1'
EXPECTED_VERSION = 'stage1a_shared_logE_global_gauge_v1'
OUT = PROJECT/'experiments/qian_pfc_timing_three_sections_1_2_7_v1'/('GPU_SMOKE' if SMOKE else 'GPU_FULL')
RECEIPT = OUT/'TIMING_SUCCESS.json'
CHECKPOINT = OUT/'model_final.pt'
EXPECTED = [(1,'GW15','UMB1117','F1b',410492),
            (2,'GW20','FB080','F1',665204),
            (7,'GW22','FB123','F2',439986)]
assert sum(x[-1] for x in EXPECTED)==1_515_682

if os.environ.get("ZOETROPE_RERUN_TIMING", "0") == "1":
    import torch
    if not (REPO/'src/flow_stp/stage1/stage1a.py').is_file():
        if (PROJECT/'src/flow_stp/stage1/stage1a.py').is_file(): REPO=PROJECT
        else: raise FileNotFoundError('Cannot locate the pinned Flow-STP source tree.')

    # --------------------------- Provenance & hardware ---------------------------
    def sha256(p):
        h=hashlib.sha256()
        with open(p,'rb') as f:
            for b in iter(lambda:f.read(4<<20),b''):h.update(b)
        return h.hexdigest()

    def git(*args):
        return subprocess.check_output(['git','-C',str(REPO),*args], text=True).strip()

    # The training source, not the notebook, is the authority on GPU support.
    commit=git('rev-parse','HEAD')
    if commit != PINNED_COMMIT:
        raise RuntimeError(f'Flow-STP commit is {commit}, not the pinned {PINNED_COMMIT}. '
                           'Use the original checkout or label a cross-version benchmark.')
    if git('status','--porcelain','--untracked-files=no','--','src/flow_stp'):
        raise RuntimeError('Tracked Flow-STP training code has local changes; refusing ambiguous timing.')
    source_file=REPO/'src/flow_stp/stage1/stage1a.py'

    if not torch.cuda.is_available():
        raise RuntimeError('CUDA is not available in this Jupyter kernel. '
            'Connect to the running Slurm GPU node and select its CUDA-enabled PyTorch kernel.')
    DEVICE=torch.device('cuda:0')
    gpu_name=torch.cuda.get_device_name(0)
    gpu_props=torch.cuda.get_device_properties(0)
    try:
        affinity=len(os.sched_getaffinity(0))
    except AttributeError:
        affinity=os.cpu_count() or 4
    cpus=max(1,min(affinity,int(os.getenv('SLURM_CPUS_PER_TASK',affinity))))
    # Set CPU helpers without hiding CUDA or changing model hyperparameters.
    torch.set_num_threads(cpus)
    try: torch.set_num_interop_threads(min(cpus,4))
    except RuntimeError: pass
    if 'SLURM_JOB_ID' not in os.environ:
        print('NOTICE: SLURM_JOB_ID unavailable; verify you are on the GPU compute node.',flush=True)
    print('GPU:', gpu_name, '| CUDA:', torch.version.cuda,
          '| GPU total GiB:',round(gpu_props.total_memory/2**30,2),flush=True)
    print('PyTorch:',torch.__version__,'| CPU workers:',cpus,'| host:',platform.node(),flush=True)

    sys.path.insert(0,str(REPO/'src'))
    from flow_stp.data.qian_pfc import QianPFCConfig, build_qian_pfc_panel, load_qian_pfc_for_stage1
    from flow_stp.stage1.stage1a import METHOD_VERSION, Stage1ARunConfig, run_stage1a
    assert METHOD_VERSION==EXPECTED_VERSION,(METHOD_VERSION,EXPECTED_VERSION)
    actual_source=Path(inspect.getfile(Stage1ARunConfig)).resolve()
    if not actual_source.is_relative_to(REPO.resolve()):
        raise RuntimeError('Stale flow_stp imported from '+str(actual_source)+'; restart the kernel.')

    cfg=Stage1ARunConfig(output_dir=OUT)  # no arbitrary optimization overrides
    assert cfg.epochs==1500 and cfg.batch_size_per_slice==1024
    assert cfg.seed==20260622 and cfg.gauge_mode=='global'
    assert cfg.coordinate_decoder=='shared_auto' and cfg.use_library_size_offset

    # The pinned repository may select CUDA automatically or expose an explicit device.
    # Prefer its native device config/signature when supported. Never silently time CPU.
    if 'device' in {f.name for f in fields(Stage1ARunConfig)}:
        cfg=replace(cfg,device='cuda:0')
    if SMOKE:
        cfg=replace(cfg,epochs=3,batch_size_per_slice=512)
    call_kwargs={}
    if 'device' in inspect.signature(run_stage1a).parameters:
        call_kwargs['device']=DEVICE

    print('Stage-1a version:',METHOD_VERSION,flush=True)
    print('Native GPU routing:',
          'Stage1ARunConfig.device' if 'device' in {f.name for f in fields(Stage1ARunConfig)}
          else ('run_stage1a(device=...)' if call_kwargs else 'trainer automatic device selection'),flush=True)
    print('Default configuration:',asdict(cfg),flush=True)
    print('Output:',OUT,flush=True)

    # -------------------------- Reuse completed run --------------------------
    if RECEIPT.is_file():
        saved=json.loads(RECEIPT.read_text())
        if saved.get('status')!='success' or saved.get('original_slice_ids')!=[1,2,7]:
            raise RuntimeError('Existing timing receipt does not match the requested cohort.')
        if saved.get('commit')!=PINNED_COMMIT or saved.get('gpu_name')!=gpu_name:
            print('NOTICE: existing successful benchmark was recorded on a different GPU or source.',flush=True)
        if not CHECKPOINT.is_file() or sha256(CHECKPOINT)!=saved['checkpoint_sha256']:
            raise RuntimeError('Completed run checkpoint checksum differs from receipt.')
        if not saved.get('gpu_peak_allocated_mib',0)>1:
            raise RuntimeError('Existing run does not establish GPU execution.')
        print('REUSED FINISHED 3-SECTION GPU FIT (no retraining):',RECEIPT,flush=True)
        train_receipt=saved
    else:
        if OUT.exists() and any(OUT.iterdir()):
            raise RuntimeError(f'Incomplete/nonempty timing run at {OUT}; inspect files '
                               'and use a new output directory rather than overwrite it.')
        for age in ('gw15','gw20','gw22'):
            if not (DATA_ROOT/f'{age}.h5ad').is_file():
                raise FileNotFoundError(DATA_ROOT/f'{age}.h5ad')

        # Verify the exact original nine-section inventory before subsetting and renumbering.
        start_loading=time.perf_counter()
        qcfg=QianPFCConfig(
            data_root=DATA_ROOT,
            targets=(('GW15',('F1a','F1b')),
                     ('GW20',('F1','F2a','F2b')),
                     ('GW22',('F1','F2','F3'))),
            expected_num_slices=9,
            layer_key_for_counts='raw',
            split_multi_sample_regions=True,
            max_cells_per_slice=(3000 if SMOKE else None),
            max_genes_for_debug=None,
            random_seed=int(cfg.seed),
        )
        panel9=build_qian_pfc_panel(qcfg).sort_values('slice_id').reset_index(drop=True)
        sel=panel9.loc[panel9.slice_id.isin([1,2,7])].copy().reset_index(drop=True)
        cohort=[(int(r.slice_id),str(r.gw),str(r['sample']),str(r.region),int(r.n_full))
                for _,r in sel.iterrows()]
        if cohort!=EXPECTED:raise RuntimeError(f'Unexpected PFC cohort: {cohort} vs {EXPECTED}')
        sel['original_slice_id']=sel.slice_id.astype(int)
        sel['slice_id']=np.arange(3,dtype=int)
        print('Exact selected sections (GW20 INCLUDED, no holdout):\n',
              sel[['original_slice_id','slice_id','gw','sample','region','time','n_full']].to_string(index=False),flush=True)

        loaded=load_qian_pfc_for_stage1(config=qcfg,panel_df=sel)
        sizes=[len(c) for c in loaded.counts_by_slice]
        expected_sizes=([3000]*3 if SMOKE else [410492,665204,439986])
        if sizes!=expected_sizes:
            raise RuntimeError(f'Unexpected loaded cell counts: {sizes}, expected {expected_sizes}')
        if len(loaded.spatial_by_slice)!=3 or len(loaded.gene_names)!=300:
            raise RuntimeError('Three-section data loader produced the wrong shape.')
        loading_seconds=time.perf_counter()-start_loading
        print('Loaded',sum(sizes),'cells x 300 genes in',round(loading_seconds,2),'s',flush=True)

        # Freeze provenance before fitting. No prior results modified.
        OUT.mkdir(parents=True,exist_ok=False)
        sel.to_csv(OUT/'training_panel.csv',index=False)
        (OUT/'gene_names.json').write_text(json.dumps(list(map(str,loaded.gene_names)),indent=2)+'\n')
        train_lock={
            'experiment':'PFC_3sections_core_fit_GPU_walltime',
            'original_slice_ids':[1,2,7],'n_cells':int(sum(sizes)),
            'n_genes':int(len(loaded.gene_names)),
            'gpu':gpu_name,'gpu_total_memory_bytes':int(gpu_props.total_memory),
            'cpu_threads':int(cpus),'host':platform.node(),
            'slurm_job_id':os.getenv('SLURM_JOB_ID'),
            'commit':PINNED_COMMIT,'stage1a_sha256':sha256(source_file),
            'method_version':METHOD_VERSION,'torch_version':str(torch.__version__),
            'cuda_version':str(torch.version.cuda),'smoke':bool(SMOKE),
            'training_cfg':{k:(str(v) if isinstance(v,Path) else v) for k,v in asdict(cfg).items()},
            'timing_definition':'torch.cuda.synchronize before and after run_stage1a; all native stages inside included',
            'not_heldout':'GW20 included in training; runtime benchmark ONLY',
        }
        (OUT/'TRAINING_LOCK.json').write_text(json.dumps(train_lock,indent=2)+'\n')

        random.seed(int(cfg.seed));np.random.seed(int(cfg.seed))
        torch.manual_seed(int(cfg.seed));torch.cuda.manual_seed_all(int(cfg.seed))
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats(0)
        print('Starting native Stage-1a GPU fit...',flush=True)
        t0=time.perf_counter()
        result=run_stage1a(
            counts_by_slice=loaded.counts_by_slice,
            spatial_by_slice=loaded.spatial_by_slice,
            panel_df=loaded.panel_df,
            gene_names=loaded.gene_names,
            obs_df=None,
            config=cfg,
            **call_kwargs,
        )
        torch.cuda.synchronize()
        fit_seconds=float(time.perf_counter()-t0)
        peak_mib=float(torch.cuda.max_memory_allocated(0)/2**20)
        reserved_mib=float(torch.cuda.max_memory_reserved(0)/2**20)
        print(f'Native fit wall clock {fit_seconds:.2f} s; CUDA peak allocated {peak_mib:.1f} MiB',flush=True)
        # Guard against a false GPU benchmark from an auto-CPU trainer.
        if peak_mib<=1:
            (OUT/'GPU_NOT_USED.json').write_text(json.dumps({
                'error':'run_stage1a did not measurably use CUDA',
                'fit_seconds':fit_seconds,'gpu_peak_allocated_mib':peak_mib},indent=2))
            raise RuntimeError('Training returned without measurable GPU allocation. '
                               'Do not report as accelerated GPU training. Inspect run_stage1a device routing.')
        if not CHECKPOINT.is_file():raise RuntimeError('Native trainer did not write model_final.pt.')
        if len(result.cells)!=sum(sizes):
            raise RuntimeError('Trained model returned wrong number of cells.')
        if not np.isfinite(result.cells['z_stage1a'].to_numpy(float)).all():
            raise RuntimeError('Nonfinite inferred coordinates.')

        train_receipt={
            'status':'success','method':'Zoetrope','original_slice_ids':[1,2,7],
            'n_cells':int(sum(sizes)),'n_genes':300,
            'commit':PINNED_COMMIT,'stage1a_sha256':sha256(source_file),
            'method_version':METHOD_VERSION,'seed':int(cfg.seed),
            'epochs':int(cfg.epochs),'batch_size_per_slice':int(cfg.batch_size_per_slice),
            'device':'cuda:0','gpu_name':gpu_name,'gpu_peak_allocated_mib':peak_mib,
            'gpu_peak_reserved_mib':reserved_mib,
            'cpu_threads':int(cpus),'torch_version':str(torch.__version__),
            'cuda_version':str(torch.version.cuda),'host':platform.node(),
            'slurm_job_id':os.getenv('SLURM_JOB_ID'),
            'preprocessing_seconds':float(loading_seconds),
            'fit_seconds':fit_seconds,
            'checkpoint_sha256':sha256(CHECKPOINT),
            'timing_definition':'synchronized wall time for run_stage1a; excludes raw data loading',
            'warning':'3-age timing run includes GW20; NOT a held-out model',
            'smoke':bool(SMOKE),
        }
        RECEIPT.write_text(json.dumps(train_receipt,indent=2)+'\n')
        print('SAVED GPU TIMING RECEIPT:',RECEIPT,flush=True)
        del result, loaded;gc.collect()

    # --------------------------- Comparator receipts ---------------------------
    # Each receipt was originally written by its author-faithful three-section notebook.
    # The scopes differ: do not claim end-to-end wall-clock equivalence.
elif not RECEIPT.is_file():
    raise FileNotFoundError(f"Copy the frozen runtime receipt to {RECEIPT} or set ZOETROPE_RERUN_TIMING=1 to time a new run.")

EXP=PROJECT/'experiments/qian_pfc_publication_4way_20261024'
RECORDS=[
    ('Zoetrope',RECEIPT,'fit_seconds','Stage-1a fit','GPU (this run)'),
    ('STORIES',EXP/'stories_slices_1_2_7_tutorial_GPU/SUCCESS.json','seconds',
     'SpaceTime.model.fit()','GPU (original run)'),
    ('stVCR',EXP/'stvcr_slices_1_2_7_official/SUCCESS.json','elapsed_seconds',
     'train_stvcr(); preproc/AE excluded','GPU (original run)'),
    ('moscot',EXP/'moscot_slices_1_2_7_AUTHOR_TUTORIAL_GPU/FIT_COMPLETE.json','fit_seconds',
     '2 spatiotemporal FGW solves','GPU (original run)'),
]
rows=[]
for method,path,key,scope,hardware in RECORDS:
    val=np.nan;status='receipt missing'
    if path.is_file():
        data=json.loads(path.read_text())
        try:
            val=float(data[key])
            if not np.isfinite(val) or val<=0: raise ValueError('nonpositive/nonfinite')
            if method=='Zoetrope' and data.get('smoke'):
                raise ValueError('smoke run cannot be compared')
            if method=='STORIES' and data.get('smoke'):
                raise ValueError('smoke run cannot be compared')
            status='measured'
        except (KeyError,TypeError,ValueError) as e:
            status='unverified value: '+str(e)
            val=np.nan
    rows.append(dict(method=method,fit_seconds=val,fit_minutes=val/60,
        receipt_status=status,model_status=(data.get('status','not recorded') if path.is_file() else 'not recorded'),
        measured_component=scope,hardware=hardware,
        receipt_path=str(path)))
frame=pd.DataFrame(rows)
if SMOKE:
    print('SMOKE MODE: timing plot skipped; run with SMOKE=False for publication.')
else:
    chart_csv=OUT/'PFC_three_age_core_training_times.csv'
    frame.to_csv(chart_csv,index=False)
    print('\nTHREE-SECTION CORE TRAINING/SOLVE TIME (not end-to-end):\n',
        frame[['method','fit_minutes','measured_component','receipt_status']].to_string(
            index=False,float_format=lambda x:f'{x:.3f}'),flush=True)

    # Publication graphic: precisely measured core-stage durations, nothing imputed.
    C={'Zoetrope':'#D62728','STORIES':'#4C78A8','stVCR':'#F58518','moscot':'#54A24B'}
    INK='#26343D'; GRID='#CFD5DB'
    mpl.rcParams.update({'font.family':'DejaVu Sans','font.size':9,
        'axes.edgecolor':INK,'axes.labelcolor':INK,'xtick.color':INK,
        'ytick.color':INK,'text.color':INK,'pdf.fonttype':42,
        'ps.fonttype':42,'svg.fonttype':'none','savefig.facecolor':'white'})
    fig,ax=plt.subplots(figsize=(6.25,3.35),facecolor='white')
    fig.subplots_adjust(left=.18,right=.97,top=.86,bottom=.33)
    order=['Zoetrope','STORIES','stVCR','moscot'];y=np.arange(4)
    show=frame.set_index('method').reindex(order)
    vals=show.fit_minutes.to_numpy(float)
    finite=np.isfinite(vals)&(vals>0)
    if finite.any():
        lo=float(np.min(vals[finite]));hi=float(np.max(vals[finite]))
        use_log=(hi/lo>30)
        for i,name in enumerate(order):
            if finite[i]:
                ax.barh(i,vals[i],height=.56,color=C[name],edgecolor='none',zorder=3)
            else:
                ax.text(.015,i,'No verified runtime receipt',va='center',ha='left',fontsize=8.5,
                        color='#7B8790',transform=ax.get_yaxis_transform())
        if use_log:
            ax.set_xscale('log');ax.set_xlim(max(lo*.55,1e-4),hi*3.5)
        else:
            ax.set_xlim(0,hi*1.35)
        for i in range(4):
            if finite[i]:
                ax.annotate(f'{vals[i]:.2f} min',xy=(vals[i],i),xytext=(5,0),
                            textcoords='offset points',va='center',ha='left',fontsize=8.7,
                            fontweight='bold',color=INK)
    else:
        ax.set_xlim(0,1)
        ax.text(.5,.5,'No verified timing receipts',ha='center',va='center',
                transform=ax.transAxes,color=INK)
    ax.set_yticks(y,order,fontweight='bold')
    ax.invert_yaxis()
    ax.set_xlabel('Recorded core training / solver wall time (minutes)',fontsize=9.2,labelpad=7)
    ax.set_title('PFC · 3 sections · 1,515,682 cells',fontsize=10.3,fontweight='bold',pad=9)
    ax.grid(axis='x',color=GRID,linewidth=.55,alpha=.65,zorder=0)
    ax.spines[['top','right','left']].set_visible(False)
    ax.tick_params(axis='y',length=0,pad=7)
    ax.tick_params(axis='x',length=3,width=.65)
    fig.text(.18,.115,'Core stages only: Zoetrope Stage-1a; STORIES fit; stVCR dynamics; moscot FGW solve.',
             fontsize=7.3,color=INK)
    moscot_state=str(show.loc['moscot','model_status'])
    caveat=('moscot solver nonconvergence reported; ' if 'nonconvergence' in moscot_state else '')
    fig.text(.18,.060,caveat+'Different preprocessing, stopping rules and hardware are not normalized.',
             fontsize=7.3,color='#6B747D')
    plotbase=OUT/'FIG_PFC_3section_runtime_GPU_Zoetrope_vs_baselines'
    for ext in ('png','pdf','svg'):
        p=plotbase.with_suffix('.'+ext)
        fig.savefig(p,dpi=450 if ext=='png' else None,facecolor='white',bbox_inches=None)
        if not p.is_file() or p.stat().st_size<1000:
            raise IOError(f'Runtime plot failed to save: {p}')
        print('SAVED',p)
    plt.show();plt.close(fig)
    print('SAVED CSV:',chart_csv)
print('\nTRAINING RECEIPT:',RECEIPT)
print('NEVER USE THIS THREE-AGE CHECKPOINT FOR A GW20-HOLDOUT BENCHMARK.')